In [1]:
import numpy as np
import pandas as pd
data = pd.read_csv("/Users/nidhishgupta/Desktop/Dream11_Fantasy Team_Predictor/data/processed/full_req_data.csv")

In [2]:
print(data.columns)
pd.set_option("display.max_columns",None)

Index(['match_id', 'player', 'fantasy_points', 'venue', 'season',
       'player_match_number', 'last3_avg_points', 'rolling_strike_rate',
       'rolling_wickets', 'opponent_avg_points', 'venue_avg_points',
       'venue_run_factor', 'batting_contribution_ratio',
       'bowling_contribution_ratio', 'last10_std_points',
       'player_consistency_index', 'form_momentum', 'bat_pos',
       'boundary_percentage', 'player_role_wicketkeeper', 'team_won_toss',
       'recent_form', 'venue_form', 'team_Deccan Chargers',
       'team_Delhi Capitals', 'team_Gujarat Lions', 'team_Gujarat Titans',
       'team_Kochi Tuskers Kerala', 'team_Kolkata Knight Riders',
       'team_Lucknow Super Giants', 'team_Mumbai Indians',
       'team_Pune Warriors', 'team_Punjab Kings', 'team_Rajasthan Royals',
       'team_Rising Pune Supergiants', 'team_Royal Challengers Bengaluru',
       'team_Sunrisers Hyderabad', 'opponent_Deccan Chargers',
       'opponent_Delhi Capitals', 'opponent_Gujarat Lions',
      

In [3]:
import joblib

model = joblib.load("/Users/nidhishgupta/Desktop/Dream11_Fantasy Team_Predictor/models/point_predicter.pkl")

In [4]:
team1 = "Chennai Super Kings"
team2 = "Mumbai Indians"

venue = "Wankhede Stadium"
team1_squad =  [
    'MS Dhoni','Ravindra Jadeja','Ruturaj Gaikwad','Deepak Chahar','Moeen Ali',
'Shivam Dube','Matheesha Pathirana','Ajinkya Rahane','Daryl Mitchell',
'Mitchell Santner','Tushar Deshpande'
]
team2_squad = ['Rohit Sharma','Ishan Kishan','Suryakumar Yadav','Hardik Pandya',
'Tim David','Jasprit Bumrah','Piyush Chawla','Tilak Varma',
'Gerald Coetzee','Romario Shepherd','Nehal Wadhera'
]
squad = team1_squad+team2_squad

In [5]:
historical_data = pd.read_csv("/Users/nidhishgupta/Desktop/Dream11_Fantasy Team_Predictor/data/processed/historical_data.csv")
historical_data['pitch_type'] = historical_data['pitch_type'].fillna('balanced')

In [6]:
historical_data.columns

Index(['match_id', 'player', 'fantasy_points', 'team', 'opponent', 'venue',
       'toss_decision', 'stage', 'season', 'player_match_number',
       'last3_avg_points', 'last5_avg_points', 'last10_avg_points',
       'rolling_strike_rate', 'rolling_wickets', 'opponent_avg_points',
       'pitch_type', 'venue_avg_points', 'venue_run_factor',
       'batting_contribution_ratio', 'bowling_contribution_ratio',
       'player_role', 'last10_std_points', 'player_consistency_index',
       'form_momentum', 'bat_pos', 'boundary_percentage', 'bat_pos_per_match',
       'player_role_wicketkeeper', 'match_month', 'team_won_toss',
       'recent_form', 'venue_form'],
      dtype='object')

In [7]:
colsToDrop =  ['last5_avg_points',
              'last10_avg_points',
              'bat_pos_per_match',
              'match_month',
              'stage'
              ]
historical_data = historical_data.drop(columns = colsToDrop)

In [8]:
from rapidfuzz import process
import re

def normalize_name(name):
    name = name.lower().strip()
    name = re.sub(r'[^a-z ]', '', name)
    name = re.sub(r'\s+', ' ', name)
    return name

# Normalize historical players
historical_data['player_key'] = historical_data['player'].apply(normalize_name)
data['player_key'] = data['player'].apply(normalize_name)
# Build player pool ONLY from squads
squad = team1_squad + team2_squad
squad_keys = {normalize_name(p): p for p in squad}

# Create reverse lookup from historical data
hist_key_to_name = dict(zip(historical_data['player_key'], historical_data['player']))

def resolve_initial(name, squad):
    parts = name.split()
    
    if len(parts) == 2 and len(parts[0]) == 1:
        initial, last = parts
        
        candidates = [
            p for p in squad
            if p.split()[-1].lower() == last.lower()
            and p[0].lower() == initial.lower()
        ]
        
        if len(candidates) == 1:
            return candidates[0]
    
    return None

def safe_map_player(name):
    name_norm = normalize_name(name)
    
    # 1️⃣ Exact match
    if name_norm in hist_key_to_name:
        return hist_key_to_name[name_norm]
    
    # 2️⃣ Initial resolution (R Sharma → Rohit Sharma)
    resolved = resolve_initial(name, squad)
    if resolved:
        resolved_norm = normalize_name(resolved)
        return hist_key_to_name.get(resolved_norm, resolved)
    
    # 3️⃣ Fuzzy match ONLY within squad
    match = process.extractOne(name, squad, score_cutoff=90)
    
    if match:
        matched_name = match[0]
        return hist_key_to_name.get(normalize_name(matched_name), matched_name)
    
    return None
    


In [9]:
data

,match_id,player,fantasy_points,venue,season,player_match_number,last3_avg_points,rolling_strike_rate,rolling_wickets,opponent_avg_points,venue_avg_points,venue_run_factor,batting_contribution_ratio,bowling_contribution_ratio,last10_std_points,player_consistency_index,form_momentum,bat_pos,boundary_percentage,player_role_wicketkeeper,team_won_toss,recent_form,venue_form,team_Deccan Chargers,team_Delhi Capitals,team_Gujarat Lions,team_Gujarat Titans,team_Kochi Tuskers Kerala,team_Kolkata Knight Riders,team_Lucknow Super Giants,team_Mumbai Indians,team_Pune Warriors,team_Punjab Kings,team_Rajasthan Royals,team_Rising Pune Supergiants,team_Royal Challengers Bengaluru,team_Sunrisers Hyderabad,opponent_Deccan Chargers,opponent_Delhi Capitals,opponent_Gujarat Lions,opponent_Gujarat Titans,opponent_Kochi Tuskers Kerala,opponent_Kolkata Knight Riders,opponent_Lucknow Super Giants,opponent_Mumbai Indians,opponent_Pune Warriors,opponent_Punjab Kings,opponent_Rajasthan Royals,opponent_Rising Pune Supergiants,opponent_Royal Challengers Bengaluru,opponent_Sunrisers Hyderabad,toss_decision_field,pitch_type_batting_friendly,pitch_type_pace_friendly,pitch_type_spin_friendly,player_role_batsman,player_role_bowler,player_key
0,548341,A Ashish Reddy,58.0,Subrata Roy Sahara Stadium,2012,1,35.672040,80.762144,0.0,39.250000,58.00,0.879923,0.000000,1.000000,0.000000,0.0,0.0,7.0,0.0,0,1,35.672040,2068.978308,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,a ashish reddy
1,548346,A Ashish Reddy,41.0,Wankhede Stadium,2012,2,35.672040,80.762144,0.0,30.000000,41.00,1.035196,0.292683,0.707317,0.000000,0.0,0.0,7.0,0.6,0,0,35.672040,1462.553632,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,True,False,False,False,False,a ashish reddy
2,548348,A Ashish Reddy,23.0,Barabati Stadium,2012,3,35.672040,80.762144,0.0,39.250000,23.00,1.027630,0.000000,1.000000,0.000000,0.0,0.0,7.0,0.0,0,1,35.672040,820.456915,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,a ashish reddy
3,548352,A Ashish Reddy,28.0,MA Chidambaram Stadium,2012,4,40.666667,80.762144,0.0,47.333333,45.00,0.984847,0.107143,0.892857,17.502381,0.0,0.0,7.0,0.0,0,0,38.668816,1740.096716,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,a ashish reddy
4,548356,A Ashish Reddy,33.0,M Chinnaswamy Stadium,2012,5,30.666667,80.762144,0.0,38.714286,31.75,1.016059,0.000000,1.000000,15.631165,0.0,0.0,7.0,0.0,0,0,32.668816,1037.234905,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,True,False,False,False,True,a ashish reddy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25120,1473478,Zeeshan Ansari,25.0,Rajiv Gandhi International Stadium,2025,6,3.333333,0.000000,0.8,18.500000,6.75,0.994802,0.000000,1.000000,36.004166,0.0,0.0,11.0,0.0,0,0,13.247204,89.418627,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,True,zeeshan ansari
25121,1473480,Zeeshan Ansari,0.0,MA Chidambaram Stadium,2025,7,11.666667,0.000000,0.4,0.000000,0.00,0.984847,0.000000,0.000000,32.204037,0.0,0.0,11.0,0.0,0,1,14.647204,0.000000,

In [13]:
historical_data

,match_id,player,fantasy_points,team,opponent,venue,toss_decision,season,player_match_number,last3_avg_points,rolling_strike_rate,rolling_wickets,opponent_avg_points,pitch_type,venue_avg_points,venue_run_factor,batting_contribution_ratio,bowling_contribution_ratio,player_role,last10_std_points,player_consistency_index,form_momentum,bat_pos,boundary_percentage,player_role_wicketkeeper,team_won_toss,recent_form,venue_form,player_key
0,548341,A Ashish Reddy,58.0,Deccan Chargers,Pune Warriors,Subrata Roy Sahara Stadium,bat,2012,1,35.672040,80.762144,0.0,39.250000,balanced,58.00,0.879923,0.000000,1.000000,bowler,0.000000,0.0,0.0,7.0,0.0,0,1,35.672040,2068.978308,a ashish reddy
1,548346,A Ashish Reddy,41.0,Deccan Chargers,Mumbai Indians,Wankhede Stadium,field,2012,2,35.672040,80.762144,0.0,30.000000,batting_friendly,41.00,1.035196,0.292683,0.707317,allrounder,0.000000,0.0,0.0,7.0,0.6,0,0,35.672040,1462.553632,a ashish reddy
2,548348,A Ashish Reddy,23.0,Deccan Chargers,Pune Warriors,Barabati Stadium,bat,2012,3,35.672040,80.762144,0.0,39.250000,balanced,23.00,1.027630,0.000000,1.000000,bowler,0.000000,0.0,0.0,7.0,0.0,0,1,35.672040,820.456915,a ashish reddy
3,548352,A Ashish Reddy,28.0,Deccan Chargers,Chennai Super Kings,MA Chidambaram Stadium,bat,2012,4,40.666667,80.762144,0.0,47.333333,spin_friendly,45.00,0.984847,0.107143,0.892857,bowler,17.502381,0.0,0.0,7.0,0.0,0,0,38.668816,1740.096716,a ashish reddy
4,548356,A Ashish Reddy,33.0,Deccan Chargers,Royal Challengers Bengaluru,M Chinnaswamy Stadium,field,2012,5,30.666667,80.762144,0.0,38.714286,batting_friendly,31.75,1.016059,0.000000,1.000000,bowler,15.631165,0.0,0.0,7.0,0.0,0,0,32.668816,1037.234905,a ashish reddy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25120,1473478,Zeeshan Ansari,25.0,Sunrisers Hyderabad,Mumbai Indians,Rajiv Gandhi International Stadium,field,2025,6,3.333333,0.000000,0.8,18.500000,balanced,6.75,0.994802,0.000000,1.000000,bowler,36.004166,0.0,0.0,11.0,0.0,0,0,13.247204,89.418627,zeeshan ansari
25121,1473480,Zeeshan Ansari,0.0,Sunrisers Hyderabad,Chennai Super Kings,MA Chidambaram Stadium,field,2025,7,11.666667,0.000000,0.4,0.000000,spin_friendly,0.00,0.984847,0.000000,0.000000,allrounder,32.204037,0.0,0.0,11.0,0.0,0,1,14.647204,0.000000,zeeshan ansari
25122,1473488,Zeeshan Ansari,23.0,Sunrisers Hyderabad,Gujarat Titans,Narendra Modi Stadium,field,2025,8,12.333333,0.000000,0.2,11.500000,balanced,23.00,1.079940,0.000000,1.000000,bowler,30.937725,0.0,0.0,11.0,0.0,0,1,13.067204,300.545692,zeeshan ansari
25123,1473492,Zeeshan Ansari,4.0,Sunrisers Hyderabad,Delhi Capitals,Rajiv Gandhi International Stadium,field,2025,9,16.000000,0.000000,0.4,44.500000,balanced,6.75,0.994802,-0.000000,1.000000,bowler,28.645618,0.0,0.0,11.0,0.0,0,1,16.647204,112.368627,zeeshan ansari


In [14]:
mapped_players_1 = [safe_map_player(p) for p in team1_squad]
mapped_players_2 = [safe_map_player(p) for p in team2_squad]
all_mapped = mapped_players_1 + mapped_players_2

unmatched = [p for p in squad if safe_map_player(p) is None]

if unmatched:
    print("⚠️ Unmatched players:", unmatched)

In [25]:
historical_data['player'].unique()

array(['A Ashish Reddy', 'A Badoni', 'A Chandila', 'A Chopra',
       'A Choudhary', 'A Dananjaya', 'A Flintoff', 'A Kamboj', 'A Kumble',
       'A Manohar', 'A Mhatre', 'A Mishra', 'A Mithun', 'A Mukund',
       'A Nehra', 'A Nel', 'A Nortje', 'A Raghuvanshi', 'A Singh',
       'A Symonds', 'A Tomar', 'A Uniyal', 'A Zampa', 'AA Bilakhia',
       'AA Chavan', 'AA Jhunjhunwala', 'AA Kazi', 'AA Kulkarni',
       'AA Noffke', 'AB Agarkar', 'AB Barath', 'AB Dinda', 'AB McDonald',
       'AB de Villiers', 'AC Blizzard', 'AC Gilchrist', 'AC Thomas',
       'AC Voges', 'AD Hales', 'AD Mascarenhas', 'AD Mathews', 'AD Nath',
       'AD Russell', 'AF Milne', 'AG Murtaza', 'AG Paunikar', 'AJ Finch',
       'AJ Hosein', 'AJ Turner', 'AJ Tye', 'AK Markram', 'AL Menaria',
       'AM Nayar', 'AM Rahane', 'AM Salvi', 'AN Ahmed', 'AN Ghosh',
       'AP Dole', 'AP Majumdar', 'AP Tare', 'AR Bawne', 'AR Patel',
       'AS Joseph', 'AS Rajpoot', 'AS Raut', 'AS Roy', 'AS Yadav',
       'AT Carey', 'AT Rayud

In [16]:
normalized_all_mapped = [normalize_name(p) for p in all_mapped]
normalized_all_mapped

['ms dhoni',
 'ravindra jadeja',
 'ruturaj gaikwad',
 'deepak chahar',
 'moeen ali',
 'shivam dube',
 'matheesha pathirana',
 'ajinkya rahane',
 'daryl mitchell',
 'mitchell santner',
 'tushar deshpande',
 'rohit sharma',
 'ishan kishan',
 'suryakumar yadav',
 'hardik pandya',
 'tim david',
 'jasprit bumrah',
 'piyush chawla',
 'tilak varma',
 'gerald coetzee',
 'romario shepherd',
 'nehal wadhera']

In [17]:
unique_venue_list_from_df = historical_data['venue'].unique()
unique_venues = list(unique_venue_list_from_df)
print(unique_venues)
def map_venue_name(name):
    
    match = get_close_matches(name, unique_venues, n=1, cutoff=0.5)
    
    if match:
        return match[0]
    else:
        return None
print(historical_data['venue'].value_counts())

['Subrata Roy Sahara Stadium', 'Wankhede Stadium', 'Barabati Stadium', 'MA Chidambaram Stadium', 'M Chinnaswamy Stadium', 'Rajiv Gandhi International Stadium', 'Arun Jaitley Stadium', 'Eden Gardens', 'Maharashtra Cricket Association Stadium', 'Sawai Mansingh Stadium', 'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium', 'Punjab Cricket Association IS Bindra Stadium', 'Brabourne Stadium', 'Dr DY Patil Sports Academy', 'Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium', 'Narendra Modi Stadium', 'Himachal Pradesh Cricket Association Stadium', 'Newlands', "St George's Park", 'Kingsmead', 'New Wanderers Stadium', 'SuperSport Park', 'Vidarbha Cricket Association Stadium, Jamtha', 'Buffalo Park', 'OUTsurance Oval', 'Nehru Stadium', 'Sheikh Zayed Stadium', 'Sharjah Cricket Stadium', 'Dubai International Cricket Stadium', 'Shaheed Veer Narayan Singh International Stadium', 'Saurashtra Cricket Association Stadium', 'Green Park', 'JSCA International Stadium Complex', 'Barsapara Cric

In [18]:
from difflib import get_close_matches
mapped_venues = map_venue_name(venue)
mapped_venues

'Wankhede Stadium'

In [21]:
players_df = data[
    data['player_key'].isin(normalized_all_mapped)
].copy()
players_df

,match_id,player,fantasy_points,venue,season,player_match_number,last3_avg_points,rolling_strike_rate,rolling_wickets,opponent_avg_points,venue_avg_points,venue_run_factor,batting_contribution_ratio,bowling_contribution_ratio,last10_std_points,player_consistency_index,form_momentum,bat_pos,boundary_percentage,player_role_wicketkeeper,team_won_toss,recent_form,venue_form,team_Deccan Chargers,team_Delhi Capitals,team_Gujarat Lions,team_Gujarat Titans,team_Kochi Tuskers Kerala,team_Kolkata Knight Riders,team_Lucknow Super Giants,team_Mumbai Indians,team_Pune Warriors,team_Punjab Kings,team_Rajasthan Royals,team_Rising Pune Supergiants,team_Royal Challengers Bengaluru,team_Sunrisers Hyderabad,opponent_Deccan Chargers,opponent_Delhi Capitals,opponent_Gujarat Lions,opponent_Gujarat Titans,opponent_Kochi Tuskers Kerala,opponent_Kolkata Knight Riders,opponent_Lucknow Super Giants,opponent_Mumbai Indians,opponent_Pune Warriors,opponent_Punjab Kings,opponent_Rajasthan Royals,opponent_Rising Pune Supergiants,opponent_Royal Challengers Bengaluru,opponent_Sunrisers Hyderabad,toss_decision_field,pitch_type_batting_friendly,pitch_type_pace_friendly,pitch_type_spin_friendly,player_role_batsman,player_role_bowler,player_key
7684,980905,Ishan Kishan,13.0,Punjab Cricket Association IS Bindra Stadium,2016,1,24.000000,81.785714,0.8,27.642857,56.333333,1.016947,1.0,0.0,36.168740,0.807327,-5.200000,2.0,0.727273,1,1,27.280000,1536.773333,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,True,False,True,False,ishan kishan
7685,980945,Ishan Kishan,2.0,Arun Jaitley Stadium,2016,2,35.672040,80.762144,0.0,44.444444,19.500000,1.031603,1.0,0.0,0.000000,0.000000,0.000000,2.0,0.000000,1,0,35.672040,695.604776,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False,True,False,ishan kishan
7686,980949,Ishan Kishan,-2.0,Maharashtra Cricket Association Stadium,2016,3,35.672040,80.762144,0.0,20.000000,17.000000,0.992518,1.0,-0.0,0.000000,0.000000,0.000000,2.0,0.000000,1,1,35.672040,606.424677,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,True,False,ishan kishan
7687,980955,Ishan Kishan,38.0,Saurashtra Cricket Association Stadium,2016,4,4.333333,80.762144,0.0,27.642857,44.750000,1.048043,1.0,0.0,7.767453,0.000000,0.000000,2.0,0.444444,1,1,16.868816,754.879512,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,True,False,False,True,False,ishan kishan
7688,980961,Ishan Kishan,2.0,Saurashtra Cricket Association Stadium,2016,5,12.666667,80.762144,0.0,44.444444,44.750000,1.048043,1.0,0.0,17.988422,0.000000,0.000000,2.0,0.000000,1,0,21.868816,978.629512,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False,True,False,ishan kishan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23066,1473493,Tilak Varma,7.0,Wankhede Stadium,2025,47,18.666667,143.084148,0.0,35.857143,33.947368,1.035196,1.0,0.0,25.954875,1.814688,-28.433333,5.0,0.000000,0,0,29.290000,994.318421,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,True,False,False,True,False,tilak varma
23067,1473501,Tilak Varma,30.0,Wankhede Stadium,2025,48,10.666667,124.463458,0.

In [22]:

players_df['player'].unique()

array(['Ishan Kishan', 'MS Dhoni', 'Tilak Varma'], dtype=object)